In [12]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage,SystemMessage,AIMessage
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder

In [7]:
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash",max_tokens=1024)

In [4]:
chat_template = ChatPromptTemplate([
    ('system', 'You are a helpful {domain} expert. Instructions: {instructions}'),  #dynamic prompt
    ('human', 'Explain in simple terms, what is {topic}')
])

In [5]:
prompt = chat_template.invoke({'domain':'cricket','topic':'LBW','instructions':'give response in less than 50 words and precise'})

In [6]:
print(prompt)

messages=[SystemMessage(content='You are a helpful cricket expert. Instructions: give response in less than 50 words and precise', additional_kwargs={}, response_metadata={}), HumanMessage(content='Explain in simple terms, what is LBW', additional_kwargs={}, response_metadata={})]


In [8]:
#storing as chat history and appending responce from model to it for making it context aware for future interactions
chat_history = prompt.messages
chat_history.append(AIMessage(content="LBW stands for Leg Before Wicket. It is a rule in cricket that allows the umpire to give a batsman out if the ball hits their leg in line with the stumps and would have gone on to hit the stumps. This rule is designed to prevent batsmen from using their legs to block the ball instead of their bat."))
chat_history

[SystemMessage(content='You are a helpful cricket expert. Instructions: give response in less than 50 words and precise', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Explain in simple terms, what is LBW', additional_kwargs={}, response_metadata={}),
 AIMessage(content='LBW stands for Leg Before Wicket. It is a rule in cricket that allows the umpire to give a batsman out if the ball hits their leg in line with the stumps and would have gone on to hit the stumps. This rule is designed to prevent batsmen from using their legs to block the ball instead of their bat.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [9]:
while True:
    user_input = input('You: ')
    chat_history.append(HumanMessage(content=user_input))
    if user_input == 'exit':
        break
    result = model.invoke(chat_history) #passing the entire chat history to model for context aware response
    chat_history.append(AIMessage(content=result.content)) #appending model response to chat history for future interactions
    print("AI: ",result.content) #printing model response

AI:  A "no ball" is an illegal delivery by the bowler. Common reasons include the bowler's front foot landing over the popping crease, or bowling a ball that's too high or dangerous. It results in a penalty run for the batting team and a "free hit" on the next delivery.


In [10]:
#saving chat history as a text file or appending if chat history already exists for future reference
with open('3_2_chat_history.txt', 'a') as f:
    for message in chat_history:
        f.write(f"{message.type}: {message.content}\n")


In [15]:
# if session starts again we can load the previous chat history and continue the conversation from there for better context awareness using messages placeholder in prompt template as
chat_template = ChatPromptTemplate([
    ('system', 'You are a helpful {domain} expert. Instructions: {instructions}'),
        MessagesPlaceholder(variable_name='3_2_chat_history'), #placeholder for chat history to maintain context
    ('human', 'Explain in simple terms, what is {topic}')
])

#chat history storing
chat_history = []
# load chat history
with open('3_2_chat_history.txt') as f:
    chat_history.extend(f.readlines())

#updating chat in chat template
result = chat_template.invoke({'domain':'cricket','topic':'LBW','instructions':'give response in less than 50 words and precise','3_2_chat_history':chat_history})

print(result.messages)


[SystemMessage(content='You are a helpful cricket expert. Instructions: give response in less than 50 words and precise', additional_kwargs={}, response_metadata={}), HumanMessage(content='system: You are a helpful cricket expert. Instructions: give response in less than 50 words and precise\n', additional_kwargs={}, response_metadata={}), HumanMessage(content='human: Explain in simple terms, what is LBW\n', additional_kwargs={}, response_metadata={}), HumanMessage(content='ai: LBW stands for Leg Before Wicket. It is a rule in cricket that allows the umpire to give a batsman out if the ball hits their leg in line with the stumps and would have gone on to hit the stumps. This rule is designed to prevent batsmen from using their legs to block the ball instead of their bat.\n', additional_kwargs={}, response_metadata={}), HumanMessage(content='human: what is no ball?\n', additional_kwargs={}, response_metadata={}), HumanMessage(content='ai: A "no ball" is an illegal delivery by the bowler